## Data Preprocessing Notebook ##
**Project**: Artificial Intelligence-Based Cell Survival Colony Counting \
**Author**: Mona Wang \
**Supervisors**: Laya Rafiee Sevyeri, Shirin A. Enger 

This Jupyter notebook will:
1. Crop and mask the background of the images
1. Unify the dimensions of the images
1. Obtain the masks of the objects in the images

In [23]:
## Toggles ##
remove_background = True
resize = True
get_mask = True

## Constants ##
MINRADIUS = 1000 # Note that these radii are for 3024 x 4032 images. The wells take up approx 1/2 of the images
MAXRADIUS = 1200

In [24]:
## Imports ##

import numpy as np
import glob
import math
import re

# Image Processing
import cv2 as cv
from skimage import morphology, img_as_ubyte

from PIL import Image

# For controlling cell execution
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

## Fix the issue of Error #15 "Initializing libiomp5md.dll, but found mk2iomp5md.dll already initialized." ##
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


## Function for Numerical Ordering ##
numbers = re.compile(r'(\d+)')
def numericalSort(value):
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts

In [25]:
## Paths ##
''' 
Variables:
    image_dir(str): directory of input data
    image_paths(list): list of all jpg files in the image_dir
    preprocess_path(str): diretory containing intermediaries
    processed_img_path (str): directory containing processed images
    processed_mask_path (str): directory containing generated masks
    processed_overlay_path (str): directory containing overlaid images
'''
image_dir = os.path.join('.', 'Data', 'HCT116_Dataset', 'img_raw') # input directory
image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
preprocess_path = os.path.join('.', 'preprocessing') # directory for the intermediaries
processed_img_path = os.path.join('.', 'Data', 'HCT116_Dataset', 'img')
processed_mask_path = os.path.join('.', 'Data', 'HCT116_Dataset', 'mask')
processed_overlay_path = os.path.join('.', 'Data', 'HCT116_Dataset', 'overlay')

## Validation ##
print(f'Number of Images: {len(image_paths)}')
print(f'One sample of Image Name: {image_paths[0]}')

Number of Images: 787
One sample of Image Name: .\Data\HCT116_Dataset\img_raw\Sample_1-1.jpg


#### Hugh Transform Background Filtering ####
Hough transform relies on the detection of circles within a given radii range. It is normal for the algorithm to accidentally detect multiple circles within that range.\
\
If any samples were not cropped correctly during the following step:
1. Move the faulty samples into the "preprocessing_reprocess" folder
1. Use the "reprocessing.ipynb" notebook to fix them.

In [27]:
%%skip_if remove_background == False

# Crop circle with OpenCV
# Documentation: https://docs.opencv.org/3.4/dd/d1a/group__imgproc__feature.html#ga47849c3be0d0406ad3ca45db65a25d2d

dir = sorted(os.listdir(image_dir), key=numericalSort)

for file in dir:
    print("now processing:", file)
    img = cv.imread(os.path.join(image_dir, file))
    img_g = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    img_g = cv.blur(img_g,(3,3)) #blurring the image


    ## Create blank mask for cropping ##
    height, width, garbage= img.shape
    mask = np.zeros((height,width), np.uint8)
    rows = img_g.shape[0]


    ## Detect circular well in the image ##
    circles = cv.HoughCircles(img_g, cv.HOUGH_GRADIENT, 1, rows / 8,
                            param1=100, param2=40, minRadius=MINRADIUS, maxRadius = MAXRADIUS)
    for i in circles[0, :]:
        center = (int(i[0]), int(i[1]))
        # circle center
        cv.circle(img_g, center, 1, (0, 100, 100), 3)
        # circle outline
        radius = int(i[2])
        cv.circle(img_g, center, radius, (255, 0, 255), 3)
        # Draw on mask
        circle_img = cv.circle(mask,(int(i[0]),int(i[1])),int(i[2]),(255,255,255),thickness=-1)


    ## Overlaying image with mask ##
    masked_data = cv.bitwise_and(img, img, mask=mask)
    _,threshold = cv.threshold(mask,1,255,cv.THRESH_BINARY)


    ## Find rectangular contour and crop the mask ##
    contours = cv.findContours(threshold,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)
    x,y,w,h = cv.boundingRect(contours[0][0])
    crop = masked_data[y:y+h,x:x+w]

    # cv.imshow("Image", img_g)
    # cv.imshow("Cropped", crop)
    # cv.waitKey(0)
    
    
    ## Save Intermediaries ##
    cv.imwrite(os.path.join(preprocess_path, file), crop)

now processing: Sample_1-1.jpg
now processing: Sample_1-2.jpg
now processing: Sample_1-3.jpg
now processing: Sample_1-4.jpg
now processing: Sample_1-5.jpg
now processing: Sample_1-6.jpg
now processing: Sample_1-7.jpg
now processing: Sample_1-8.jpg
now processing: Sample_1-9.jpg
now processing: Sample_1-10.jpg
now processing: Sample_1-11.jpg
now processing: Sample_1-12.jpg
now processing: Sample_1-13.jpg
now processing: Sample_1-14.jpg
now processing: Sample_1-15.jpg
now processing: Sample_1-16.jpg
now processing: Sample_1-17.jpg
now processing: Sample_1-18.jpg
now processing: Sample_2-1.jpg
now processing: Sample_2-2.jpg
now processing: Sample_2-3.jpg
now processing: Sample_2-4.jpg
now processing: Sample_2-5.jpg
now processing: Sample_2-6.jpg
now processing: Sample_2-7.jpg
now processing: Sample_2-8.jpg
now processing: Sample_2-9.jpg
now processing: Sample_2-10.jpg
now processing: Sample_2-11.jpg
now processing: Sample_2-12.jpg
now processing: Sample_2-13.jpg
now processing: Sample_2-1

#### Resizing Images ####

In [ ]:
## Unifying Size ##
%%skip_if resize == False

dir = sorted(os.listdir(preprocess_path), key=numericalSort)
widths = []
heights = []

## Find avg width and height of the cropped images ##
for files in dir:
    filepath = preprocess_path + files
    img = cv.imread(filepath)
    widths.append(img.shape[0])
    heights.append(img.shape[1])
    
AVG_WIDTH = round(sum(widths) / len(widths))
AVG_HEIGHT = round(sum(heights) / len(heights))

## Resize images ##
for files in dir:
    filepath = image_dir + files
    img = Image.open(filepath)
    img = img.resize((AVG_WIDTH,AVG_HEIGHT))
    img.save(processed_img_path + '\\' + files, "JPEG", quality=100, subsampling=0)

#### Obtaining Masks ####

In [ ]:
# Binary segmentation <- https://stackoverflow.com/questions/72171280/how-to-segment-object-from-the-background-of-an-image-python-opencv

dir = os.listdir(processed_img_path)

for image in dir:
    if os.path.isfile(processed_img_path+'\\'+image):
        print(image)
        img = cv.imread(processed_img_path + '\\' + image) # Sample_1-1.jpg for testing
        img = cv.bilateralFilter(img, 5, 75, 75)


        ## Manual Thresholding ##
        low = np.array([0, 0, 1])
        high = np.array([169, 171, 168]) #[188, 194, 197]) #rbg
        mask = cv.inRange(img, low, high)
        

        ## Removal of Noises ##
        '''
        Variables:
            inter(CV_8U): intermediate masks resulted from morphological transformations
            inter_cropped(CV_8U): intermediate masks cropped with a circular mask
        '''
        kernel = cv.getStructuringElement(cv.MORPH_RECT, (3, 3)) # returns structuring element of specified size and shape
        smoothKernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))

        inter = cv.morphologyEx(mask,cv.MORPH_CLOSE, kernel, iterations=5)
        inter = cv.morphologyEx(inter, cv.MORPH_CLOSE, smoothKernel, iterations=5)
        inter = cv.morphologyEx(inter, cv.MORPH_OPEN, smoothKernel, iterations=5)
        inter = morphology.remove_small_holes(inter, area_threshold=3000)
        inter = img_as_ubyte(inter)


        ## Crop the mask with a circle for removing edge noise ##
        img_height, img_width, garbage = img.shape
        circular = np.zeros((img_height,img_width), dtype=np.uint8)
        circleMask = cv.circle(circular,(math.floor(img_height / 2), math.floor(img_width / 2)), math.floor(img_height / 2)-15,(255,255,255), thickness=-1)
        inter_cropped = cv.bitwise_or(inter, inter, mask=circleMask)


        ## Remove non-colonies from the mask ##
        '''
        Variables:
            finalMask(UnsignedByte): processed mask
            finalMasked(UnsignedByte): original sample with processed mask overlaid
        '''
        nb_blobs, mask_separated_blobs, stats, _ = cv.connectedComponentsWithStats(inter_cropped)
        sizes = stats[:, cv.CC_STAT_AREA]
        finalMask = np.zeros_like(mask_separated_blobs)

        for i in range(1, nb_blobs):
            if sizes[i] >= 400: #exclude small objects <---- tune this hyperparameter for batch less than 80%
                finalMask[mask_separated_blobs == i] = 255
        finalMask = img_as_ubyte(finalMask)
        finalMasked = cv.bitwise_and(img, img, mask=finalMask)

        # cv.imshow("Smoothed", finalMask)
        # cv.waitKey(0)
        # cv.imshow("Masked", result)
        # cv.waitKey(0)


        ## Saving processed masks ##
        cv.imwrite(processed_mask_path + image, finalMask)
        cv.imwrite(processed_overlay_path + image, finalMasked)

#### Validation

In [12]:
imgs = glob.glob(os.path.join(processed_img_path, '*.jpg'))
masks = glob.glob(os.path.join(processed_mask_path, '*.jpg'))
overlays = glob.glob(os.path.join(processed_overlay_path, '*.jpg'))
print(f'Number of Images: {len(imgs)}')
print(f'Number of Masks: {len(masks)}')
print(f'Number of Overlays: {len(overlays)}')

Number of Images: 787
Number of Masks: 787
Number of Overlays: 787
